# 🌿 Plant Disease Detection — Training Notebook (Colab)

Trains a MobileNetV2 CNN on the **PlantVillage** dataset and exports a **TFLite** model for the Flutter app.

**Stack:** Pandas (EDA) · Matplotlib (visualization) · TensorFlow/Keras (model) · scikit-learn (evaluation) · TFLite (mobile export)

> ⚡ **Before running:** Runtime → Change runtime type → **T4 GPU**

**Outputs (downloaded at the end):**
- `plant_disease_model.tflite` → copy to `flutter_app/assets/models/`
- `labels.txt` → copy to `flutter_app/assets/models/`

## 1. Setup & Dataset Download (Kaggle)

In [ ]:
import tensorflow as tf
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

**Get your Kaggle API key:** [kaggle.com/settings](https://www.kaggle.com/settings) → *Create New Token* → downloads `kaggle.json`.

Run the next cell and upload that file when prompted.

In [ ]:
from google.colab import files
import os

# Upload kaggle.json
uploaded = files.upload()

os.makedirs('/root/.kaggle', exist_ok=True)
os.replace('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print("Kaggle credentials configured ✓")

In [ ]:
# Download and unzip PlantVillage (~330 MB)
!kaggle datasets download -d emmarex/plantdisease -p /content --unzip -q

# The dataset unzips to /content/PlantVillage
DATA_DIR = '/content/PlantVillage'
!ls {DATA_DIR} | head -20

## 2. EDA — Pandas + Matplotlib

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

rows = []
for class_dir in sorted(Path(DATA_DIR).iterdir()):
    if class_dir.is_dir():
        rows.append({'class': class_dir.name,
                     'count': len(list(class_dir.glob('*')))})

df = pd.DataFrame(rows)
print(f"Total classes : {len(df)}")
print(f"Total images  : {df['count'].sum()}")
print(f"Imbalance ratio (max/min): {df['count'].max() / df['count'].min():.2f}")
df.sort_values('count', ascending=False)

In [ ]:
# Class distribution
plt.figure(figsize=(10, 7))
d = df.sort_values('count')
plt.barh(d['class'], d['count'], color='seagreen')
plt.xlabel('Number of images')
plt.title('PlantVillage Class Distribution')
plt.tight_layout()
plt.show()

## 3. Load Datasets (80/20 train/val split)

In [ ]:
IMG_SIZE = 224   # must match inputSize in tflite_service.dart
BATCH_SIZE = 32
SEED = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset='training', seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset='validation', seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)
print(f"\n{NUM_CLASSES} classes:")
class_names

In [ ]:
# Sample images
plt.figure(figsize=(12, 12))
for images, labels in train_ds.take(1):
    for i in range(16):
        plt.subplot(4, 4, i + 1)
        plt.imshow(images[i].numpy().astype('uint8'))
        plt.title(class_names[labels[i]], fontsize=8)
        plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Performance: cache + prefetch
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

## 4. Build Model — MobileNetV2 Transfer Learning

Normalization (`preprocess_input`) is **inside the graph**, so the Flutter app feeds raw 0-255 pixels.

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomContrast(0.1),
])

base = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False, weights='imagenet')
base.trainable = False  # Phase 1: frozen backbone

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)  # scales to [-1, 1]
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

## 5. Phase 1 — Train with Frozen Backbone

In [ ]:
EPOCHS = 10

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.5),
]

history1 = model.fit(train_ds, validation_data=val_ds,
                     epochs=EPOCHS, callbacks=callbacks)

## 6. Phase 2 — Fine-tune Top Layers

In [ ]:
FINE_TUNE_EPOCHS = 5

base.trainable = True
for layer in base.layers[:-40]:   # only top 40 layers trainable
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),  # low LR!
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

history2 = model.fit(train_ds, validation_data=val_ds,
                     epochs=FINE_TUNE_EPOCHS)

In [ ]:
# Training curves (both phases combined)
h = {k: history1.history[k] + history2.history[k]
     for k in history1.history}
hist_df = pd.DataFrame(h)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist_df['accuracy'], label='train')
axes[0].plot(hist_df['val_accuracy'], label='val')
axes[0].axvline(len(history1.history['accuracy']) - 1, ls='--', c='gray')
axes[0].set_title('Accuracy'); axes[0].legend()
axes[1].plot(hist_df['loss'], label='train')
axes[1].plot(hist_df['val_loss'], label='val')
axes[1].axvline(len(history1.history['loss']) - 1, ls='--', c='gray')
axes[1].set_title('Loss'); axes[1].legend()
plt.tight_layout()
plt.show()

print(f"Final val accuracy: {hist_df['val_accuracy'].iloc[-1]:.4f}")

## 7. Evaluation — scikit-learn

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_true, y_pred = [], []
for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(12, 10))
plt.imshow(cm, cmap='Greens')
plt.colorbar()
ticks = np.arange(NUM_CLASSES)
plt.xticks(ticks, class_names, rotation=90, fontsize=7)
plt.yticks(ticks, class_names, fontsize=7)
plt.xlabel('Predicted'); plt.ylabel('True')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

## 8. Export TFLite Model (float16 quantized)

This is the file the Flutter app loads.

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_model = converter.convert()

with open('plant_disease_model.tflite', 'wb') as f:
    f.write(tflite_model)

with open('labels.txt', 'w') as f:
    f.write('\n'.join(class_names))

print(f"TFLite size: {len(tflite_model)/1e6:.1f} MB")
print(f"Labels: {NUM_CLASSES} classes written to labels.txt")

In [ ]:
# Sanity check: run one inference with the TFLite interpreter
interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()
inp = interpreter.get_input_details()[0]
out = interpreter.get_output_details()[0]
print("Input :", inp['shape'], inp['dtype'])
print("Output:", out['shape'], out['dtype'])

# test with one validation image (raw 0-255 floats, like the Flutter app sends)
for images, labels in val_ds.take(1):
    test_img = images[0:1].numpy().astype(np.float32)
    interpreter.set_tensor(inp['index'], test_img)
    interpreter.invoke()
    scores = interpreter.get_tensor(out['index'])[0]
    print(f"\nTrue     : {class_names[labels[0]]}")
    print(f"Predicted: {class_names[np.argmax(scores)]} ({scores.max():.2%})")

## 9. Download Files for the Flutter App

Copy both files into `flutter_app/assets/models/` then run `flutter pub get && flutter run`.

In [ ]:
from google.colab import files
files.download('plant_disease_model.tflite')
files.download('labels.txt')